PART 1: `sample_config.yaml` -- The Competition Pipeline Config
   This file lives at `aic/aic_engine/config/sample_config.yaml`. It's the
   MASTER CONFIGURATION that `aic_engine` reads to set up and run competition
   trails. It defines EVERYTHING about what the scene looks like and what the
   robot needs to do.

SECTION 1: `scoring.topics` 
   This lists every ROS 2 topic the scoring system subscribes to. These are
   the signals the competition evaluator watches:

```yaml
scoring:
  topics:
    - topic: 
      name: "/joint_states"         # Where is each robot arm right now?
      type: "sensor_msgs/msg/JointState"
    - topic: 
      name: "/tf"
      ...
                                    # Live TF transforms (robot + gripper poses)
                                    
                                    # Cable link poses (where each cable segment
                                    # is)
                                    
                                    # Did you touch something forbidden?
                                    # Force/torque at the insertion point
                                    # What commands did your model send?
                                    # Cartesian pose commands
                                    # "SC plug inserted!" event
                                    # Controller internal state
```

   WHY THIS MATTERS: The evaluator records all of these during a trial. If your#
   robot touches an off-limit surface or exceed force limits, you lose points.
   The `insertion_event` topic is how the system knows a plug was successfully
   inserted.


---
Section 2: `task_board_limits`


```yaml
    min_translation: ...  # Minimum translation from center along rail...
    max_translations: ... # Maximum translation from center along rail (in meters)
```
   ...

   THIS IS THE LEGAL RANGE OF WHERE components can be placed along each rail
   type. The translation value is the OFFSET FROM RAIL CENTER ALONG THE Y-AXIS.
   This is what `randomize_board.py` should respect -- it defines the limit of 
   where NIC cards, SC ports, and mount fixtures can slide.


---
SECTION 3: `trails` -- The big one
   Each trail defines a COMPLETE SCENE ++ TASK. This is how the competition
   works. 

```yaml
trails:
  trail_1:                  <-- First scenario
    scene:                  <-- What the world looks like
      task_board:           <-- Board configuration
        pose:               <-- Where the board sits in the world
        nic_rail_0:         <-- What's on each rail
        ...
      cables:               <-- Cable assembly config
    tasks:                  <-- What the robot must accomplish
      task_1:               <-- insert this plug into that port    

```


Trail 1 scene breakdown:

```yaml
  task_board:
    pose:
      x: 0.15, y: -0.2, z: 1.14      # Board world position
      yaw: 3.1415                    # Rotated ~180 degree (facing the robot)

    nic_rail_0:
      entity_present: True           # There IS a NIC card here
      entity_name: "nic_card_0"      # Its name for TF/scoring
      entity_pose:
        translation: 0.036           # Slide 3.6cm from rail center
        yaw: 0.0                     # No rotation

    nic_rail_1 through 4:
      entity_present: False           # Empty rails

    sc_rail_0:
      entity_present: True
      entity_name: "sc_mount_0"       # NOTE: SC port on SC rail
      entity_pose:
        translation: 0.042
        yaw: 0.1                      # Slight way! Not always zero

    lc_mount_rail_0:
      entity_present: True
      entity_name: "lc_mount_0"       # LC mount fixture
      entity_pose:
        translation: 0.02

    sfp_mount_rail_0:
      entity_present: True
      entity_name: "sfp_mount_0"      # SFP mount fixture

    sc_mount_rail_0:
      entity_present: True
      entity_name: "sc_mount_0"       # SC mount fixture
```

   KEY INSIGHT YOU MENTIONED MISSING: The config shows that MOUNT FIXTURES CAN
   HAVE NON-ZERO YAW (`sc_rail_0` has `yaw: 0.1`). Our randomiser currently
   forces zero yaw -- which is correct for mounts (screwed to rails), but SC
   ports on SC rails CAN have slight yaw.


---
CABLE CONFIGURATION:
```yaml
        cables:                     
          cable_0:
            pose:
              gripper_offset:           # Where cable attaches to gripper
                x: 0.0
                y: 0.015385
                z: 0.04245
              roll: 0.4432
              pitch: -0.4838
              yaw: 1.3303
            attach_cable_to_gripper: True       # Cable starts held by robot
            cable_type: "sfp_sc_cable" # [sfp_sc_cable, sfp_sc_cable_reversed]
                                                # Which cable model (SFP on one end, SC on other)
```

   THIS IS CRITICAL: The cable is pre-attached to the gripper! The robot starts
   HOLDING the cable. The plug is at the cable end. This is why `aic_engine.cpp`
   doesn't spawn plugs separately -- the plug is physically connected to the 
   cable, which is held by the gripper.


---
TASK DEFINITION
```yaml
    tasks:
      task_1:
        cable_type: "sfp_sc"
        cable_name: "cable_0"
        plug_type: "sfp"
        plug_name: "sfp_tip"
        port_type: "sfp"
        port_name: "sfp_port_0"
        target_module_name: "nic_card_mount_0"
        time_limit: 180
```
   TRAIL 3 is different -- it changes the board pose (x=0.17, yaw=3.0), uses
   different rails, and tasks the robot with inserting the SC end 
   (`cable_type: "sfp_sc_cable_reversed"`) into an SC port.


---
SECTION 4: `robot` 
```yaml
robot:
  home_joint_positions:
    shoulder_pan_joint: -0.1597
    shoulder_lift_joint: -1.3542
    elbow_joint: -1.6648
    wrist_1_joint: -1.6933
    wrist_2_joint: 1.5710
    wrist_3_joint: 1.4110

```
   The robot's starting pose for each trail. This is where the UR5e arm begins
   before attempting insertion.



---
SUMMARY OF WHAT THIS CONFIG TELLS US

   1. The competition runs multiple trails with different board configurations
      and tasks
   2. Components slide along rails with bounded translation ranges
   3. The cable is pre-attached to the gripper -- plugs are cable endpoints, not
      standalone objects
   4. Mount fixtures CAN have slight yaw on SC rails (0.1 rad = 5.7 degree)
   5. Each trail has a specific insertion task -- insert plug X into port Y
      within time limit
   6. THE BOARD POSE CHANGES BETWEEN TRAILS -- the robot must handle different
      positions/orientations